<a href="https://colab.research.google.com/github/CPTR295/NLP-Basic-Using-Pytorch/blob/main/Classification_Yelp_Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import collections
import numpy as np
import pandas as pd
import re
import json
import string
import os

from argparse import Namespace
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm_notebook

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("omkarsabnis/yelp-reviews-dataset")

print("Path to dataset files:", path)

100%|██████████| 3.49M/3.49M [00:00<00:00, 191MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/omkarsabnis/yelp-reviews-dataset/versions/1


In [ ]:
!mv '/root/.cache/kagglehub/datasets/omkarsabnis/yelp-reviews-dataset/versions/1' '/content/'

In [ ]:
full_df = pd.read_csv('/content/1/yelp.csv')
full_df.head()

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1,2,0
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0


In [ ]:
len(full_df)

10000

In [ ]:
df = full_df[['text','stars']]

In [ ]:
df.head()

,text,stars
0,My wife took me here on my birthday for breakf...,5
1,I have no idea why some people give bad review...,5
2,love the gyro plate. Rice is so good and I als...,4
3,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",5
4,General Manager Scott Petello is a good egg!!!...,5


In [ ]:
df["stars"] = (df["stars"] >= 4).astype(int)

/tmp/ipykernel_354/3249322986.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["stars"] = (df["stars"] >= 4).astype(int)


In [ ]:
df.head()

,text,stars
0,My wife took me here on my birthday for breakf...,1
1,I have no idea why some people give bad review...,1
2,love the gyro plate. Rice is so good and I als...,1
3,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",1
4,General Manager Scott Petello is a good egg!!!...,1


In [ ]:
import re
def preprocess_text(text):
  text = text.lower()
  text = re.sub(r"([.,!?])", r" \1 ", text)
  text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
  return text

In [ ]:
df.text = df.text.apply(preprocess_text)

/tmp/ipykernel_354/3916824729.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.text = df.text.apply(preprocess_text)


In [ ]:
df.head()

,text,stars
0,my wife took me here on my birthday for breakf...,1
1,i have no idea why some people give bad review...,1
2,love the gyro plate . rice is so good and i al...,1
3,"rosie , dakota , and i love chaparral dog park...",1
4,general manager scott petello is a good egg ! ...,1


In [ ]:
args = Namespace(
    raw_train_dataset_csv="/content/1/yelp.csv",
    train_proportion=0.6,
    val_proportion=0.3,
    test_proportion=0.1,
    output_munged_csv="/content/yelp.csv",
    seed=1337
)

In [ ]:
set(df.stars)
by_rating = collections.defaultdict(list)
for _, row in df.iterrows():
    by_rating[row.stars].append(row.to_dict())

In [ ]:
final_list = []
np.random.seed(args.seed)

for _, item_list in sorted(by_rating.items()):

    np.random.shuffle(item_list)

    n_total = len(item_list)
    n_train = int(args.train_proportion * n_total)
    n_val = int(args.val_proportion * n_total)
    n_test = int(args.test_proportion * n_total)

    # Give data point a split attribute
    for item in item_list[:n_train]:
        item['split'] = 'train'

    for item in item_list[n_train:n_train+n_val]:
        item['split'] = 'val'

    for item in item_list[n_train+n_val:n_train+n_val+n_test]:
        item['split'] = 'test'

    # Add to final list
    final_list.extend(item_list)

In [ ]:
final_reviews = pd.DataFrame(final_list)

In [ ]:
final_reviews.split.value_counts()

,count
split,
train,5999
val,2999
test,999


In [ ]:
final_reviews.stars.value_counts()

,count
stars,
1,6863
0,3137


In [ ]:
final_reviews.head()

,text,stars,split
0,i had seen these around before but never stopp...,0,train
1,first off made a reservation and arrived to be...,0,train
2,this location was a groupon find . . . won t b...,0,train
3,"wow , what a blast from the past ! i hadn t ea...",0,train
4,our fearless critical mass organizer arranged ...,0,train


In [ ]:
final_reviews['rating'] = final_reviews.stars.apply({0: 'negative', 1: 'positive'}.get)

In [ ]:
final_reviews.head()

,text,stars,split,rating
0,i had seen these around before but never stopp...,0,train,negative
1,first off made a reservation and arrived to be...,0,train,negative
2,this location was a groupon find . . . won t b...,0,train,negative
3,"wow , what a blast from the past ! i hadn t ea...",0,train,negative
4,our fearless critical mass organizer arranged ...,0,train,negative


In [ ]:
final_reviews = final_reviews.drop(columns='stars')

In [ ]:
final_reviews.head()

,text,split,rating
0,i had seen these around before but never stopp...,train,negative
1,first off made a reservation and arrived to be...,train,negative
2,this location was a groupon find . . . won t b...,train,negative
3,"wow , what a blast from the past ! i hadn t ea...",train,negative
4,our fearless critical mass organizer arranged ...,train,negative


In [ ]:
final_reviews.to_csv(args.output_munged_csv, index=False)

CLASSIFICATION

In [ ]:
class Vocabulary(object):
  #Class to process text and extract vocabulary for mapping
  def __init__(self,token_to_idx=None,add_unk=True,unk_token="<UNK>"):
     #token_to_idx(Dict) - pre-existing map of token to indices
     #add_unk(bool) - a flag to add_unk
     #unk_token(str) - unk token to add into vocabulary
     if token_to_idx is None:
      token_to_idx = {}
     self._token_to_idx = token_to_idx
     self._idx_to_token = {idx:token for token,idx in self._token_to_idx.items()}
     self._add_unk=add_unk
     self._unk_token = unk_token
     self.unk_index = -1
     if add_unk:
      self.unk_index = self.add_token(unk_token)

  def to_serializable(self):
    #Returns a dict that can be serialzed
    return {'token_to_idx':self._token_to_idx,
            'add_unk':self._add_unk,
            'unk_token':self._unk_token}

  @classmethod
  def from_serializable(cls,contents):
    #Instantiates the vocabulary from serialized dict
    return cls(**contents)

  def add_token(self,token):
    #Add item to vocabulary and return its corresponding index
    if token in self._token_to_idx:
      index = self._token_to_idx[token]
    else:
      index = len(self._token_to_idx)
      self._token_to_idx[token] = index
      self._idx_to_token[index] = token
    return index

  def add_many(self,tokens):
    #Add list of items to vocabulary and return their corresponding indices
    return [self.add_token(token) for token in tokens]

  def lookup_token(self,token):
    #Retrieve the index associated with the token or the UNK
    if self.unk_index >= 0:
      return self._token_to_idx.get(token,self.unk_index)
    else:
      return self._token_to_idx[token]

  def lookup_index(self,index):
    #Retrieve the token associated with the index or UNK
    if index not in self._idx_to_token:
      raise KeyError("the index (%d) is not in the vocabulary" % index)
    return self._idx_to_token[index]

  def __str__(self):
    return "<Vocabulary(size=%d)>" % len(self)

  def __len__(self):
    return len(self._token_to_idx)


In [ ]:
class ReviewVectorizer(object):
  #The vectorizer which coordinates the vocabualry
  def __init__(self,review_vocab,rating_vocab):
    #review_vocab - maps words to integer
    #rating_vocab - maps label to integer
    self.review_vocab = review_vocab
    self.rating_vocab = rating_vocab

  def vectorize(self,review):
    #Create a collapsed one-hot vector for the review
    one_hot = np.zeros(len(self.review_vocab),dtype=np.float32)
    for token in review.split(" "):
      if token not in string.punctuation:
        one_hot[self.review_vocab.lookup_token(token)] =1
    return one_hot

  @classmethod
  def from_dataframe(cls,review_df,cutoff=25):
    #Intiates the vectorfrom the dataset dataframe
    review_vocab = Vocabulary(add_unk=True)
    rating_vocab = Vocabulary(add_unk=False)

    for rating in sorted(set(review_df.rating)):
      rating_vocab.add_token(rating)

    word_counts = collections.Counter()
    for review in review_df.text:
      for word in review.split(" "):
        if word not in string.punctuation:
          word_counts[word] += 1
    for word,count in word_counts.items():
      if count > cutoff:
        review_vocab.add_token(word)
    return cls(review_vocab,rating_vocab)

  def to_serializable(self):
    return {'review_vocab':self.review_vocab.to_serializable(),
            'rating_vocab':self.rating_vocab.to_serializable()}
  @classmethod
  def from_serializable(cls,contents):
    review_vocab = Vocabulary.from_serializable(contents['review_vocab'])
    rating_vocab = Vocabulary.from_serializable(contents['rating_vocab'])
    return cls(review_vocab=review_vocab,rating_vocab=rating_vocab)



In [ ]:
class ReviewDataset(Dataset):
  def __init__(self,review_df,vectorizer):
    self.review_df = review_df
    self._vectorizer = vectorizer

    self.train_df = self.review_df[self.review_df.split=='train']
    self.train_size = len(self.train_df)

    self.val_df = self.review_df[self.review_df.split=='val']
    self.validation_size = len(self.val_df)

    self.test_df = self.review_df[self.review_df.split=='test']
    self.test_size = len(self.test_df)

    self.lookup_dict = {'train':(self.train_df,self.train_size),
                      'val':(self.val_df,self.validation_size),
                      'test':(self.test_df,self.test_size)}
    self.set_split('train')

  @classmethod
  def load_dataset_and_make_vectorizer(cls, review_csv):
      """Load dataset and make a new vectorizer from scratch

      Args:
          review_csv (str): location of the dataset
      Returns:
          an instance of ReviewDataset
      """
      review_df = pd.read_csv(review_csv)
      train_review_df = review_df[review_df.split=='train']
      return cls(review_df, ReviewVectorizer.from_dataframe(train_review_df))

  @classmethod
  def load_dataset_and_load_vectorizer(cls, review_csv, vectorizer_filepath):
      """Load dataset and the corresponding vectorizer.
      Used in the case in the vectorizer has been cached for re-use

      Args:
          review_csv (str): location of the dataset
          vectorizer_filepath (str): location of the saved vectorizer
      Returns:
          an instance of ReviewDataset
      """
      review_df = pd.read_csv(review_csv)
      vectorizer = cls.load_vectorizer_only(vectorizer_filepath)
      return cls(review_df, vectorizer)

  def get_vectorizer(self):
    return self._vectorizer

  def set_split(self,split="train"):
    self._target_split = split
    self._target_df,self._target_size = self.lookup_dict[split]

  @staticmethod
  def load_vectorizer_only(vectorizer_filepath):
    with open(vectorizer_filepath) as fp:
      return ReviewVectorizer.from_serializable(json.load(fp))

  def save_vectorizer(self,vectorizer_filepath):
    with open(vectorizer_filepath,"w") as fp:
      json.dump(self._vectorizer.to_serializable(),fp)

  def __len__(self):
    return self._target_size

  def __getitem__(self, index):
    row = self._target_df.iloc[index]
    review_vector = self._vectorizer.vectorize(row.text)
    rating_index = self._vectorizer.rating_vocab.lookup_token(row.rating)
    return {'x_data':review_vector,'y_target':rating_index}

  def get_num_batches(self,batch_size):
    return len(self)

In [ ]:
def generate_batches(dataset,batch_size,shuffle=True,drop_last=True,device="cpu"):
  dataloader = DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last)
  for data_dict in dataloader:
    out_data_dict = {}
    for name,tensor in data_dict.items():
      out_data_dict[name]=data_dict[name].to(device)
    yield out_data_dict

In [ ]:
class ReviewClassifier(nn.Module):
  #A single perceptron based classifier
  def __init__(self,num_features):
    super(ReviewClassifier,self).__init__()
    self.fc1 = nn.Linear(in_features=num_features,out_features=1)

  def forward(self,x_in,apply_sigmoid=False):
    #from tensor(batch,num_featrues) -> (batch,)
    y_out = self.fc1(x_in).squeeze()
    if apply_sigmoid:
      y_out = torch.sigmoid(y_out)
    return y_out

In [ ]:
def make_train_state(args):
  return {'stop_early':False,
          'early_stopping_step':0,
          'early_stopping_best_val':1e8,
          'learning_rate':args.learning_rate,
          'epoch_index':0,
          'train_loss':[],
          'train_acc':[],
          'val_loss':[],
          'val_acc':[],
          'test_loss':-1,
          'test_acc':-1,
          'model_filename':args.model_state_file
          }


In [ ]:
def update_train_state(args,model,train_state):
  if train_state['epoch_index']==0:
    torch.save(model.state_dict(),train_state['model_filename'])
    train_state['stop_early']=False
  elif train_state['epoch_index']>=1:
    loss_tm1,loss_t = train_state['val_loss'][-2:]
    if loss_t>=train_state['early_stopping_best_val']:
      train_state['early_stopping_step']+=1
    else:
      if loss_t<train_state['early_stopping_best_val']:
        torch.save(model.state_dict(),train_state['model_filename'])
      train_state['early_stopping']=0
    train_state['stop_early'] = train_state['early_stopping_step']>=args.early_stopping
  return train_state

In [ ]:
def compute_accuracy(y_pred,y_target):
  y_target=y_target.cpu()
  y_pred_indices = (torch.sigmoid(y_pred)>0.5).cpu().long()
  n_correct = torch.eq(y_pred_indices,y_target).sum().item()
  return n_correct/len(y_pred_indices)*100

In [ ]:
def set_seed_everywhere(seed,cuda):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if cuda:
    torch.cuda.manual_seed(seed)

In [ ]:
def handle_dirs(dirpath):
  if not os.path.exists(dirpath):
    os.makedirs(dirpath)

In [ ]:
args = Namespace(
    frequency_cutoff=25,
    model_state_file = 'model.pth',
    review_csv='/content/yelp.csv',
    save_dir='/content',
    vectorizer_file='vectorizer.json',
    batch_size=128,
    early_stopping=3,
    learning_rate=0.001,
    num_epochs=100,
    seed=1337,
    catch_keyboard_interrupt=True,
    cuda=True,
    expand_filepaths_to_save_dir=True,
    reload_from_files=False
)

In [ ]:
if args.expand_filepaths_to_save_dir:
  args.vectorizer_file = os.path.join(args.save_dir,args.vectorizer_file)
  args.model_state_file = os.path.join(args.save_dir,args.model_state_file)
  print("Expanded filepaths: ")
  print("\t{}".format(args.vectorizer_file))
  print("\t{}".format(args.model_state_file))

Expanded filepaths: 
	/content/vectorizer.json
	/content/model.pth


In [ ]:
if not torch.cuda.is_available():
  args.cuda = False
  print("Cuda Not Present")

In [ ]:
print('Using CUDA: {}'.format(args.cuda))

Using CUDA: True


In [ ]:
args.device = torch.device('cuda' if args.cuda else 'cpu')
set_seed_everywhere(args.seed,args.cuda)
handle_dirs(args.save_dir)

Initialization

In [ ]:
if args.reload_from_files:
  print('Loading dataset and vectorizer')
  dataset = ReviewDataset.load_dataset_and_make_vectorizer(args.review_csv)
else:
  print('Loading dataset and creating vectorizer')
  dataset = ReviewDataset.load_dataset_and_make_vectorizer(args.review_csv)
  dataset.save_vectorizer(args.vectorizer_file)

Loading dataset and creating vectorizer


In [ ]:
vectorizer = dataset.get_vectorizer()
classifier = ReviewClassifier(num_features=len(vectorizer.review_vocab))

Traing Loop

In [ ]:
classifier = classifier.to(args.device)

loss_func = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(classifier.parameters(),lr=args.learning_rate)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,mode='min',factor=0.5,patience=1)

train_state = make_train_state(args)

In [ ]:
try:
  for epoch_index in range(args.num_epochs):
    train_state['epoch_index'] = epoch_index
    #Training
    dataset.set_split('train')
    batch_generator = generate_batches(dataset=dataset,batch_size=args.batch_size,device=args.device)
    running_loss = 0.0
    running_acc = 0.0
    classifier.train()
    for batch_index,batch_dict in enumerate(batch_generator,start=1):
      optimizer.zero_grad()
      y_pred = classifier(x_in=batch_dict['x_data'].float())
      loss = loss_func(y_pred,batch_dict['y_target'].float())

      loss.backward()
      optimizer.step()
      running_loss += (loss.item() - running_loss) / batch_index

      acc = compute_accuracy(y_pred, batch_dict['y_target'])
      running_acc += (acc - running_acc) / batch_index
    train_state['train_loss'].append(running_loss)
    train_state['train_acc'].append(running_acc)
    #Validation
    dataset.set_split('val')
    batch_generator = generate_batches(dataset,batch_size=args.batch_size,device=args.device)
    running_loss = 0.0
    running_acc = 0.0
    classifier.eval()
    with torch.no_grad():
      for batch_index,batch_dict in enumerate(batch_generator,start=1):
        y_pred = classifier(x_in=batch_dict['x_data'].float())
        loss = loss_func(y_pred,batch_dict['y_target'].float())
        running_loss += (loss.item() - running_loss) / batch_index

        acc = compute_accuracy(y_pred, batch_dict['y_target'])
        running_acc += (acc - running_acc) / batch_index
    train_state['val_loss'].append(running_loss)
    train_state['val_acc'].append(running_acc)
    train_state = update_train_state(args=args,model=classifier,train_state=train_state)
    scheduler.step(train_state['val_loss'][-1])
    if train_state['stop_early']:
      break
    print(
        f"Epoch {epoch_index+1:3d}/{args.num_epochs} | "
        f"Train Loss: {train_state['train_loss'][-1]:.4f} | "
        f"Train Acc: {train_state['train_acc'][-1]:6.2f}% | "
        f"Val Loss: {train_state['val_loss'][-1]:.4f} | "
        f"Val Acc: {train_state['val_acc'][-1]:6.2f}%"
    )
except KeyboardInterrupt:
  print('Exiting loop')

Epoch   1/100 | Train Loss: 0.3005 | Train Acc:  89.39% | Val Loss: 0.4002 | Val Acc:  83.05%
Epoch   2/100 | Train Loss: 0.2972 | Train Acc:  89.28% | Val Loss: 0.3987 | Val Acc:  82.88%
Epoch   3/100 | Train Loss: 0.2923 | Train Acc:  89.45% | Val Loss: 0.3988 | Val Acc:  82.85%
Epoch   4/100 | Train Loss: 0.2892 | Train Acc:  89.59% | Val Loss: 0.3989 | Val Acc:  82.88%
Epoch   5/100 | Train Loss: 0.2847 | Train Acc:  89.64% | Val Loss: 0.3978 | Val Acc:  83.02%
Epoch   6/100 | Train Loss: 0.2821 | Train Acc:  89.98% | Val Loss: 0.3979 | Val Acc:  82.98%
Epoch   7/100 | Train Loss: 0.2812 | Train Acc:  89.96% | Val Loss: 0.3945 | Val Acc:  83.02%
Epoch   8/100 | Train Loss: 0.2793 | Train Acc:  90.05% | Val Loss: 0.3975 | Val Acc:  83.12%
Epoch   9/100 | Train Loss: 0.2775 | Train Acc:  90.03% | Val Loss: 0.3946 | Val Acc:  83.22%
Epoch  10/100 | Train Loss: 0.2752 | Train Acc:  90.29% | Val Loss: 0.3973 | Val Acc:  82.85%
Epoch  11/100 | Train Loss: 0.2752 | Train Acc:  90.20% | Va

In [ ]:
# compute the loss & accuracy on the test set using the best available model

classifier.load_state_dict(torch.load(train_state['model_filename']))
classifier = classifier.to(args.device)

dataset.set_split('test')
batch_generator = generate_batches(dataset,
                                   batch_size=args.batch_size,
                                   device=args.device)
running_loss = 0.
running_acc = 0.
classifier.eval()

for batch_index, batch_dict in enumerate(batch_generator):
    # compute the output
    y_pred = classifier(x_in=batch_dict['x_data'].float())

    # compute the loss
    loss = loss_func(y_pred, batch_dict['y_target'].float())
    loss_t = loss.item()
    running_loss += (loss_t - running_loss) / (batch_index + 1)

    # compute the accuracy
    acc_t = compute_accuracy(y_pred, batch_dict['y_target'])
    running_acc += (acc_t - running_acc) / (batch_index + 1)

train_state['test_loss'] = running_loss
train_state['test_acc'] = running_acc

In [ ]:
print("Test loss: {:.3f}".format(train_state['test_loss']))
print("Test Accuracy: {:.2f}".format(train_state['test_acc']))

Test loss: 0.389
Test Accuracy: 83.71


In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"([.,!?])", r" \1 ", text)
    text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
    return text

In [ ]:
def predict_rating(review, classifier, vectorizer, decision_threshold=0.5):
    """Predict the rating of a review

    Args:
        review (str): the text of the review
        classifier (ReviewClassifier): the trained model
        vectorizer (ReviewVectorizer): the corresponding vectorizer
        decision_threshold (float): The numerical boundary which separates the rating classes
    """
    review = preprocess_text(review)

    vectorized_review = torch.tensor(vectorizer.vectorize(review))
    result = classifier(vectorized_review.view(1, -1))

    probability_value = F.sigmoid(result).item()
    index = 1
    if probability_value < decision_threshold:
        index = 0

    return vectorizer.rating_vocab.lookup_index(index)

In [ ]:
test_review = "this is a pretty awesome book"

classifier = classifier.cpu()
prediction = predict_rating(test_review, classifier, vectorizer, decision_threshold=0.5)
print("{} -> {}".format(test_review, prediction))

this is a pretty awesome book -> positive


In [ ]:
classifier.fc1.weight.shape

torch.Size([1, 2333])

In [ ]:
# Sort weights
fc1_weights = classifier.fc1.weight.detach()[0]
_, indices = torch.sort(fc1_weights, dim=0, descending=True)
indices = indices.numpy().tolist()

# Top 20 words
print("Influential words in Positive Reviews:")
print("--------------------------------------")
for i in range(20):
    print(vectorizer.review_vocab.lookup_index(indices[i]))

print("====\n\n\n")

# Top 20 negative words
print("Influential words in Negative Reviews:")
print("--------------------------------------")
indices.reverse()
for i in range(20):
    print(vectorizer.review_vocab.lookup_index(indices[i]))

Influential words in Positive Reviews:
--------------------------------------
delicious
wonderful
highly
awesome
amazing
great
community
beat
favorites
best
incredible
recommend
perfect
fantastic
celebrate
extensive
private
pepperoni
mesa
sexy
====



Influential words in Negative Reviews:
--------------------------------------
overpriced
mediocre
ok
horrible
sucked
awful
okay
bland
rude
bad
elsewhere
nothing
not
lacking
waste
dry
worst
disappointment
flavorless
foot
